# J2 — Evaluation and hyper-parameter tuning of the fraud model

**Objective.** Show that the fraud model is actually good, using metrics that fit the problem, and that tuning it with a
real search helps (or does not) compared with the untuned model. Then check the pipeline for data leakage and say what
was ruled out.

**Prior models.** The fraud classifier from the earlier tracks (XGBoost on the engineered features) and the logistic
regression baseline from J1. Same data, same 70/15/15 stratified split, same features as before, so the numbers connect
to [01_supervised_fraud.ipynb](01_supervised_fraud.ipynb) and [05_final_comparison.ipynb](05_final_comparison.ipynb).

**Protocol, fixed before running anything.**
1. *Untuned* means the library defaults (XGBoost: depth 6, learning rate 0.3, 100 trees, no class weighting; logistic
   regression: C = 1, no class weighting). *Tuned* means the best configuration of a real search. Neither uses early
   stopping, so the only difference between the two is the hyper-parameters.
2. The searches see **training rows only**, a stratified 30% sample of them, scored by 3-fold stratified CV on PR-AUC.
3. Tuned and untuned are compared by stratified 5-fold CV on the training set (to see the spread), and then once on the
   test set at a threshold chosen on the validation set. The test difference gets a paired bootstrap interval.
4. Whatever comes out is reported. Earlier tracks found that tuning moves this model very little, so "no real difference"
   is an acceptable result.

| § | Section |
|---|---|
| 1 | Setup |
| 2 | Why these metrics |
| 3 | The searches |
| 4 | Tuned against untuned |
| 5 | Leakage checks |

## 1. Setup

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score
from sklearn.model_selection import cross_validate
from xgboost import XGBClassifier

ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT))

from fraud.data import SEED, load_paysim, scope_transfer_cashout, split_70_15_15
from fraud.features import make_features
from fraud.leakage import (check_rows_within, features_ignore_label, permuted_label_scores, row_hashes, rows_shared,
                           univariate_pr_auc)
from fraud.metrics import WeightedAP, best_f1_threshold, evaluate
from fraud.model import make_logreg
from fraud.search import grid_search, random_search, stratified_folds, stratified_subsample

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
J2_DIR = ROOT / "reports" / "j2"
J2_DIR.mkdir(parents=True, exist_ok=True)

d = scope_transfer_cashout(load_paysim())
X, RAW, ENG = make_features(d)
y = d["isFraud"].to_numpy()
train_idx, val_idx, test_idx = split_70_15_15(y)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
X_tr, X_va, X_te = X.iloc[train_idx][ENG], X.iloc[val_idx][ENG], X.iloc[test_idx][ENG]

# The same split as the earlier tracks: the test set is 415,562 rows with 1,232 frauds.
assert len(test_idx) == 415_562 and y_test.sum() == 1_232
print(f"train {len(train_idx):,} rows ({y_train.sum():,} frauds), validation {len(val_idx):,} ({y_val.sum():,}), "
      f"test {len(test_idx):,} ({y_test.sum():,}); fraud rate {y.mean():.3%}")

train 1,939,286 rows (5,749 frauds), validation 415,561 (1,232), test 415,562 (1,232); fraud rate 0.296%


## 2. Why these metrics

The data is 0.3% fraud, so the choice of metric decides whether the evaluation says anything. The evidence for each
choice comes from this project's own results.

In [2]:
ablation = pd.read_csv(ROOT / "reports" / "ablation.csv")
lr_raw = ablation[(ablation["model"] == "Logistic regression") & (ablation["features"] == "raw")].iloc[0]
always_no = 1 - y_test.mean()

why = pd.DataFrame([
    {"metric": "Accuracy", "role": "not used",
     "why": f"A model that never flags anything is {always_no:.2%} accurate on the test set and catches no fraud at all."},
    {"metric": "ROC-AUC", "role": "reported, not the headline",
     "why": f"Logistic regression on the raw features has ROC-AUC {lr_raw['ROC-AUC']:.3f}, which looks excellent, but its PR-AUC "
            f"is {lr_raw['PR-AUC']:.3f}: about half its alerts would be wrong. ROC-AUC is flattered by the huge number of legitimate rows."},
    {"metric": "PR-AUC (average precision)", "role": "headline",
     "why": f"It asks how precise the alerts are at each level of recall, which is what an analyst feels. The naive score equals the "
            f"fraud rate ({y_test.mean():.3%}), so the scale is honest."},
    {"metric": "Precision, recall, F1, missed frauds, false alarms", "role": "operating point",
     "why": "These need a threshold, chosen on the validation set and applied unchanged to the test set. The counts say how many "
            "frauds are missed and how many alerts are wasted."},
    {"metric": "Stratified cross-validation", "role": "how the search and the CV are run",
     "why": f"With 0.3% fraud, plain folds could hold very few frauds. Stratified folds keep the rate in each fold "
            f"(about {y_train.mean():.2%} here), so every fold has frauds to learn from and be scored on."},
])
display(why.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

metric,role,why
Accuracy,not used,A model that never flags anything is 99.70% accurate on the test set and catches no fraud at all.
ROC-AUC,"reported, not the headline","Logistic regression on the raw features has ROC-AUC 0.992, which looks excellent, but its PR-AUC is 0.549: about half its alerts would be wrong. ROC-AUC is flattered by the huge number of legitimate rows."
PR-AUC (average precision),headline,"It asks how precise the alerts are at each level of recall, which is what an analyst feels. The naive score equals the fraud rate (0.296%), so the scale is honest."
"Precision, recall, F1, missed frauds, false alarms",operating point,"These need a threshold, chosen on the validation set and applied unchanged to the test set. The counts say how many frauds are missed and how many alerts are wasted."
Stratified cross-validation,how the search and the CV are run,"With 0.3% fraud, plain folds could hold very few frauds. Stratified folds keep the rate in each fold (about 0.30% here), so every fold has frauds to learn from and be scored on."


## 3. The searches

### 3.1 The search sample
Both searches run on a stratified 30% sample of the **training** rows. The check below turns the claim "the search never
saw validation or test rows" into an assertion.

In [3]:
search_pos = stratified_subsample(y_train, 0.30)                  # positions inside the training set
check_rows_within(train_idx[search_pos], train_idx, "search rows")   # raises if any row is outside the training set
X_search, y_search = X_tr.iloc[search_pos], y_train[search_pos]
ratio = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"search sample: {len(X_search):,} rows, {int(y_search.sum()):,} frauds ({y_search.mean():.3%}); all of them training rows")

search sample: 581,785 rows, 1,725 frauds (0.297%); all of them training rows


### 3.2 XGBoost: random search
`RandomizedSearchCV`, 40 configurations drawn at random from the space below, each scored by 3-fold stratified CV on
PR-AUC. The space is wider than the one used in J1 (it adds the number of trees, `gamma` and `reg_lambda`).

In [4]:
XGB_BASE = dict(tree_method="hist", eval_metric="aucpr", n_jobs=-1, random_state=SEED)
xgb_space = {
    "n_estimators": [100, 200, 400],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1, 0.2, 0.3],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 5, 10],
    "gamma": [0, 1, 5],
    "reg_lambda": [1, 5, 10],
    "scale_pos_weight": [1.0, round(float(np.sqrt(ratio)), 1), round(ratio, 1)],
}
t0 = time.time()
best_xgb, xgb_table = random_search(XGBClassifier(**XGB_BASE), xgb_space, X_search, y_search, n_iter=40)
xgb_search_seconds = time.time() - t0
print(f"40 configurations x 3 folds in {xgb_search_seconds:.0f}s")
display(xgb_table.head(8).style.format({"cv_PR-AUC": "{:.5f}", "cv_std": "{:.5f}", "mean_fit_seconds": "{:.1f}"}).hide(axis="index"))
print(f"CV PR-AUC across all 40 configurations: from {xgb_table['cv_PR-AUC'].min():.5f} to {xgb_table['cv_PR-AUC'].max():.5f}")
print("best:", best_xgb)

40 configurations x 3 folds in 475s


rank,subsample,scale_pos_weight,reg_lambda,n_estimators,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree,cv_PR-AUC,cv_std,mean_fit_seconds
1,0.800000,336.300000,1,100,1,5,0.050000,0,1.000000,0.99869,0.00111,1.9
2,1.000000,336.300000,10,100,5,4,0.300000,5,1.000000,0.99863,0.00127,1.3
3,0.800000,18.300000,1,100,1,4,0.300000,5,1.000000,0.99856,0.00081,1.8
4,0.600000,18.300000,10,100,1,6,0.100000,1,0.800000,0.99847,0.00109,1.9
5,0.800000,336.300000,10,200,1,3,0.050000,0,0.600000,0.99843,0.00129,3.9
6,1.000000,18.300000,1,100,5,4,0.050000,0,0.800000,0.99838,0.00050,1.2
7,0.600000,336.300000,5,100,1,6,0.030000,0,1.000000,0.99836,0.00099,1.9
8,1.000000,1.000000,5,400,1,4,0.100000,0,1.000000,0.99836,0.00133,4.3


CV PR-AUC across all 40 configurations: from 0.98983 to 0.99869
best: {'subsample': 0.8, 'scale_pos_weight': 336.3, 'reg_lambda': 1, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 1.0}


### 3.3 Logistic regression: grid search
`GridSearchCV` over the regularisation strength and class weighting (every one of the 10 combinations), same folds and
metric. It is cheap, and it shows the grid method as well as the random one.

In [5]:
lr_grid = {"logisticregression__C": [0.01, 0.1, 1, 10, 100], "logisticregression__class_weight": [None, "balanced"]}
t0 = time.time()
best_lr_raw, lr_table = grid_search(make_logreg(), lr_grid, X_search, y_search)
print(f"{len(lr_table)} combinations x 3 folds in {time.time() - t0:.0f}s")
best_lr = {name.split("__")[1]: value for name, value in best_lr_raw.items()}
display(lr_table.style.format({"cv_PR-AUC": "{:.5f}", "cv_std": "{:.5f}", "mean_fit_seconds": "{:.1f}"}).hide(axis="index"))
print("best:", best_lr)

search_results = pd.concat([xgb_table.assign(model="XGBoost (random search)"), lr_table.assign(model="Logistic regression (grid search)")],
                           ignore_index=True)
search_results.insert(0, "model", search_results.pop("model"))
search_results.to_csv(J2_DIR / "search_results.csv", index=False)

10 combinations x 3 folds in 32s


rank,logisticregression__C,logisticregression__class_weight,cv_PR-AUC,cv_std,mean_fit_seconds
1,0.010000,nan,0.99350,0.00070,0.7
2,0.100000,nan,0.99309,0.00111,0.5
3,1.000000,nan,0.99298,0.00120,0.7
4,10.000000,nan,0.99297,0.00122,0.6
5,100.000000,nan,0.99297,0.00122,0.5
6,0.010000,balanced,0.99109,0.00395,1.1
7,0.100000,balanced,0.99056,0.00357,1.2
8,1.000000,balanced,0.98464,0.00478,1.3
9,10.000000,balanced,0.97737,0.00953,1.3
10,100.000000,balanced,0.97570,0.01087,1.3


best: {'C': 0.01, 'class_weight': None}


## 4. Tuned against untuned

Four models, same features and split: XGBoost and logistic regression, each untuned (library defaults) and tuned (best
configuration from the search above).

In [6]:
models = {
    "XGBoost, untuned": lambda: XGBClassifier(**XGB_BASE),
    "XGBoost, tuned": lambda: XGBClassifier(**XGB_BASE, **best_xgb),
    "Logistic regression, untuned": lambda: make_logreg(),
    "Logistic regression, tuned": lambda: make_logreg(**best_lr),
}
settings = {"XGBoost, untuned": "library defaults", "XGBoost, tuned": str(best_xgb),
            "Logistic regression, untuned": "C=1, no class weight", "Logistic regression, tuned": str(best_lr)}

### 4.1 Stratified 5-fold cross-validation on the training set
This is the spread check: if the tuned and untuned scores differ by less than their fold-to-fold spread, the difference is
not worth much.

In [7]:
cv_rows = []
for name, make in models.items():
    t0 = time.time()
    res = cross_validate(make(), X_tr, y_train, cv=stratified_folds(5), scoring={"PR-AUC": "average_precision", "ROC-AUC": "roc_auc"}, n_jobs=1)
    cv_rows.append({"model": name, "cv PR-AUC": res["test_PR-AUC"].mean(), "cv PR-AUC std": res["test_PR-AUC"].std(ddof=1),
                    "cv ROC-AUC": res["test_ROC-AUC"].mean(), "cv ROC-AUC std": res["test_ROC-AUC"].std(ddof=1),
                    "cv seconds": time.time() - t0})
    print(f"{name}: {time.time() - t0:.0f}s")
cv_tbl = pd.DataFrame(cv_rows).set_index("model")
display(cv_tbl.style.format({"cv PR-AUC": "{:.5f}", "cv PR-AUC std": "{:.5f}", "cv ROC-AUC": "{:.5f}", "cv ROC-AUC std": "{:.5f}", "cv seconds": "{:.0f}"}))

XGBoost, untuned: 32s


XGBoost, tuned: 47s


Logistic regression, untuned: 13s


Logistic regression, tuned: 14s


,cv PR-AUC,cv PR-AUC std,cv ROC-AUC,cv ROC-AUC std,cv seconds
model,,,,,
"XGBoost, untuned",0.99636,0.00275,0.99871,0.00115,32
"XGBoost, tuned",0.99802,0.00128,0.99951,0.00053,47
"Logistic regression, untuned",0.99253,0.00261,0.99637,0.00126,13
"Logistic regression, tuned",0.99269,0.00250,0.99627,0.00126,14


### 4.2 Test set, used once
Each model is fitted on the whole training set. Its F1 threshold is chosen on the validation set and applied unchanged to
the test set.

In [8]:
fitted, test_scores, rows = {}, {}, []
for name, make in models.items():
    model = make()
    t0 = time.time()
    model.fit(X_tr, y_train)
    secs = time.time() - t0
    thr = best_f1_threshold(y_val, model.predict_proba(X_va)[:, 1])
    score = model.predict_proba(X_te)[:, 1]
    fitted[name], test_scores[name] = model, score
    rows.append({**evaluate(name, y_test, score, thr, "engineered", secs), "threshold": thr})
    print(f"{name}: fitted in {secs:.0f}s")
test_tbl = pd.DataFrame(rows).set_index("model")
display(test_tbl[["PR-AUC", "ROC-AUC", "precision", "recall", "F1", "FN", "FP", "fit_seconds"]]
        .style.format({"PR-AUC": "{:.4f}", "ROC-AUC": "{:.4f}", "precision": "{:.3f}", "recall": "{:.3f}", "F1": "{:.3f}",
                       "FN": "{:,}", "FP": "{:,}", "fit_seconds": "{:.0f}"}))

XGBoost, untuned: fitted in 8s


XGBoost, tuned: fitted in 11s


Logistic regression, untuned: fitted in 2s


Logistic regression, tuned: fitted in 2s


,PR-AUC,ROC-AUC,precision,recall,F1,FN,FP,fit_seconds
model,,,,,,,,
"XGBoost, untuned",0.9975,0.9983,1.000,0.994,0.997,7,0,8
"XGBoost, tuned",0.9975,0.9987,1.000,0.997,0.998,4,0,11
"Logistic regression, untuned",0.9921,0.9954,0.990,0.982,0.986,22,12,2
"Logistic regression, tuned",0.9922,0.9954,0.985,0.984,0.985,20,18,2


### 4.3 Is the tuned-minus-untuned difference real? Paired bootstrap
The test rows are re-weighted 1,000 times with Poisson(1) weights, the same weights for both models of a pair, and the 95%
interval of the PR-AUC difference is reported (the method of notebook 05).

In [9]:
rng = np.random.default_rng(SEED)
B = 1000
pairs = {"XGBoost": ("XGBoost, untuned", "XGBoost, tuned"), "Logistic regression": ("Logistic regression, untuned", "Logistic regression, tuned")}
aps = {name: WeightedAP(y_test, s) for name, s in test_scores.items()}
ones = np.ones(len(y_test))
for name, s in test_scores.items():
    assert abs(aps[name](ones) - average_precision_score(y_test, s)) < 1e-9     # same check as in J1
diffs = {k: np.empty(B) for k in pairs}
for b in range(B):
    w = rng.poisson(1.0, len(y_test)).astype(float)
    for k, (untuned, tuned) in pairs.items():
        diffs[k][b] = aps[tuned](w) - aps[untuned](w)

boot_rows = []
for k, (untuned, tuned) in pairs.items():
    lo, hi = np.percentile(diffs[k], [2.5, 97.5])
    boot_rows.append({"model": k, "untuned PR-AUC": test_tbl.loc[untuned, "PR-AUC"], "tuned PR-AUC": test_tbl.loc[tuned, "PR-AUC"],
                      "difference": test_tbl.loc[tuned, "PR-AUC"] - test_tbl.loc[untuned, "PR-AUC"], "95% low": lo, "95% high": hi,
                      "tuned higher in": float((diffs[k] > 0).mean()), "verdict": "real difference" if (lo > 0 or hi < 0) else "not distinguishable from noise"})
boot = pd.DataFrame(boot_rows)
display(boot.style.format({"untuned PR-AUC": "{:.4f}", "tuned PR-AUC": "{:.4f}", "difference": "{:+.4f}", "95% low": "{:+.4f}", "95% high": "{:+.4f}", "tuned higher in": "{:.1%}"}).hide(axis="index"))

combined = (cv_tbl.join(test_tbl[["PR-AUC", "ROC-AUC", "precision", "recall", "F1", "FN", "FP", "fit_seconds", "threshold"]])
            .rename(columns={"PR-AUC": "test PR-AUC", "ROC-AUC": "test ROC-AUC", "precision": "test precision", "recall": "test recall",
                             "F1": "test F1", "FN": "test missed frauds", "FP": "test false alarms", "fit_seconds": "fit seconds"}))
combined.insert(0, "settings", pd.Series(settings))
combined.reset_index().to_csv(J2_DIR / "tuned_vs_untuned.csv", index=False)
boot.to_csv(J2_DIR / "bootstrap.csv", index=False)

model,untuned PR-AUC,tuned PR-AUC,difference,95% low,95% high,tuned higher in,verdict
XGBoost,0.9975,0.9975,-0.0000,-0.0001,+0.0000,33.8%,not distinguishable from noise
Logistic regression,0.9921,0.9922,+0.0001,-0.0000,+0.0004,96.5%,not distinguishable from noise


### 4.4 Reading the result

- **Tuning did not change how well the model ranks frauds.** On the test set, tuned and untuned XGBoost have the same
  PR-AUC (0.9975 both), and the bootstrap interval for the difference straddles zero. The tuned logistic regression is
  also within noise of the untuned one, though only just (its interval runs from about zero to +0.0004).
- **In cross-validation the tuned XGBoost looks better (0.9980 against 0.9964), but the untuned model's fold-to-fold
  spread is 0.0028,** larger than the gap. Together with the test result, that is not evidence of a real gain in ranking.
- **A bias in the cross-validation numbers.** The 5-fold CV runs on the whole training set, which contains the sample
  the search used to pick the tuned settings, so the tuned models' CV scores are somewhat optimistic. The test set, which
  the search never saw, is the fair comparison, and it shows no difference.
- **Tuning did change XGBoost's operating point.** At the threshold picked on validation, the untuned model misses 7
  frauds and the tuned one 4, both with no false alarms. The untuned model's best-F1 threshold is extreme (0.9995)
  because without class weighting its scores are squeezed towards zero. These counts are small, so 7 against 4 is
  suggestive and has not been tested.
- **What the search picked.** The best XGBoost settings use the full class weight (336), depth 5, learning rate 0.05 and
  100 trees. Forty configurations spread from 0.9898 to 0.9987 in search CV, and the top few are within fold noise of each
  other.
- **A correction to an earlier result.** The grid search switched class weighting off for logistic regression:
  "balanced" was worse at every value of C. J1's logistic regression used "balanced", so it was a weaker baseline than it
  needed to be. With its settings tuned it scores 0.992 on the test set, not 0.970, and XGBoost's lead over it falls from
  about 0.028 to about 0.005. XGBoost is still ahead; the margin reported in J1 was too wide.

## 5. Leakage checks

Each check asks one question and states what passing does and does not show. The results are collected in a table at the
end and written up in [reports/j2/leakage_note.md](../reports/j2/leakage_note.md).

### 5.1 Are the features independent of the label?
Shuffle the label column in the raw data, rebuild the features, and compare. If any feature used the label, it would change.

In [10]:
label_free = features_ignore_label(make_features, d, "isFraud")
print("features identical after shuffling the label column:", label_free)
assert label_free

features identical after shuffling the label column: True


### 5.2 Is any single feature suspiciously good on its own?
A screen, not a proof: a column that almost gives the label away needs an explanation (is it computed from the row's own
fields, and is it available when the prediction is made?).

In [11]:
screen = univariate_pr_auc(X_tr, y_train).to_frame("PR-AUC of the feature alone")
screen["naive PR-AUC"] = y_train.mean()
display(screen.style.format("{:.4f}"))

,PR-AUC of the feature alone,naive PR-AUC
oldbalanceOrg,0.2896,0.0030
errorBalanceDest,0.2115,0.0030
newbalanceDest,0.1276,0.0030
hour,0.0489,0.0030
errorBalanceOrig,0.0299,0.0030
amount,0.0193,0.0030
newbalanceOrig,0.0117,0.0030
oldbalanceDest,0.0112,0.0030
isTransfer,0.0053,0.0030
origEmptied,0.0032,0.0030


### 5.3 Do identical rows sit in both train and test?
Identical feature rows get identical predictions, so a test row with a twin in the training set can be answered from
memory. The check counts them and recomputes PR-AUC without them.

In [12]:
shared = rows_shared(X_tr, X_te)
train_hash = row_hashes(X_tr)
fraud_hash = set(train_hash[y_train == 1])
test_hash = row_hashes(X_te)
twin_is_fraud = np.array([h in fraud_hash for h in test_hash[shared]])
dup = pd.DataFrame({
    "test rows with a training twin": [int(shared.sum())],
    "share of test rows": [shared.mean()],
    "test frauds with a training twin": [int((shared & (y_test == 1)).sum())],
    "share of test frauds": [(shared & (y_test == 1)).sum() / y_test.sum()],
    "of those, twin is a training fraud": [int(twin_is_fraud[y_test[shared] == 1].sum())],
})
display(dup.style.format({"test rows with a training twin": "{:,}", "share of test rows": "{:.2%}", "test frauds with a training twin": "{:,}",
                          "share of test frauds": "{:.2%}", "of those, twin is a training fraud": "{:,}"}).hide(axis="index"))

rows = []
for name in ("XGBoost, untuned", "XGBoost, tuned"):
    full = average_precision_score(y_test, test_scores[name])
    clean = average_precision_score(y_test[~shared], test_scores[name][~shared])
    rows.append({"model": name, "PR-AUC, all test rows": full, "PR-AUC, twin-free test rows": clean, "change": clean - full})
twin_tbl = pd.DataFrame(rows)
display(twin_tbl.style.format({"PR-AUC, all test rows": "{:.4f}", "PR-AUC, twin-free test rows": "{:.4f}", "change": "{:+.4f}"}).hide(axis="index"))

test rows with a training twin,share of test rows,test frauds with a training twin,share of test frauds,"of those, twin is a training fraud"
11,0.00%,11,0.89%,11


model,"PR-AUC, all test rows","PR-AUC, twin-free test rows",change
"XGBoost, untuned",0.9975,0.9975,-0.0000
"XGBoost, tuned",0.9975,0.9975,-0.0000


### 5.4 With the training labels shuffled, does the test score stay far below the real one?
The whole tuned pipeline is refitted on shuffled training labels and scored against the true test labels. With nothing
real to learn, the score should be far below the real one. A route by which test information reaches the score without
passing through the training labels would push it up towards the real score.

Average precision under shuffled labels is a noisy statistic on this data (the seeds below show how noisy), so the
criterion is not "exactly the fraud rate" but "a small fraction of the real score" (below 10% of it). **This criterion was set
after a first run showed scores above the fraud rate, so unlike the rules in notebook 03 it was not fixed in advance.** A leak would
be expected to put the score near the real value, so the bound is generous, but it is a judgment. As a control the same is done
with a depth-2 model, which has far less capacity to fit noise.

In [13]:
def tuned_fit_predict(X_train, y_train_perm, X_test):
    return XGBClassifier(**XGB_BASE, **best_xgb).fit(X_train, y_train_perm).predict_proba(X_test)[:, 1]


def shallow_fit_predict(X_train, y_train_perm, X_test):
    return XGBClassifier(**XGB_BASE, **{**best_xgb, "max_depth": 2, "scale_pos_weight": 1.0}).fit(X_train, y_train_perm).predict_proba(X_test)[:, 1]


SHUFFLE_SEEDS = (SEED, 1, 2, 3, 4)
t0 = time.time()
perm = permuted_label_scores(tuned_fit_predict, X_tr, y_train, X_te, y_test, seeds=SHUFFLE_SEEDS)
perm_shallow = permuted_label_scores(shallow_fit_predict, X_tr, y_train, X_te, y_test, seeds=SHUFFLE_SEEDS)
real_ap = average_precision_score(y_test, test_scores["XGBoost, tuned"])
print(f"{time.time() - t0:.0f}s")
perm_tbl = pd.DataFrame({
    "run": ["tuned model, real labels"] + [f"tuned model, shuffled labels (seed {s})" for s in SHUFFLE_SEEDS]
           + [f"depth-2 control, shuffled labels (seed {s})" for s in SHUFFLE_SEEDS],
    "test PR-AUC": [real_ap] + perm + perm_shallow})
display(perm_tbl.style.format({"test PR-AUC": "{:.4f}"}).hide(axis="index"))
print(f"chance level (the fraud rate): {y_test.mean():.4f}")
print(f"highest shuffled-label score: tuned {max(perm):.4f}, control {max(perm_shallow):.4f}; that is {max(perm + perm_shallow) / real_ap:.1%} of the real score")

# Under a shuffled fit, do the true frauds score higher than legitimate rows? (a leak would make them.)
p_shuffled = tuned_fit_predict(X_tr, np.random.default_rng(SEED).permutation(y_train), X_te)
print(f"under one shuffled fit: median score of true frauds {np.median(p_shuffled[y_test == 1]):.3f}, "
      f"of legitimate rows {np.median(p_shuffled[y_test == 0]):.3f}")

89s


run,test PR-AUC
"tuned model, real labels",0.9975
"tuned model, shuffled labels (seed 42)",0.0051
"tuned model, shuffled labels (seed 1)",0.0060
"tuned model, shuffled labels (seed 2)",0.0156
"tuned model, shuffled labels (seed 3)",0.0020
"tuned model, shuffled labels (seed 4)",0.0098
"depth-2 control, shuffled labels (seed 42)",0.0229
"depth-2 control, shuffled labels (seed 1)",0.0433
"depth-2 control, shuffled labels (seed 2)",0.0068
"depth-2 control, shuffled labels (seed 3)",0.0027


chance level (the fraud rate): 0.0030
highest shuffled-label score: tuned 0.0156, control 0.0538; that is 5.4% of the real score


under one shuffled fit: median score of true frauds 0.411, of legitimate rows 0.495


### 5.5 Did tuning, thresholds and scaling stay inside their lanes?

In [14]:
lr_pipeline = fitted["Logistic regression, tuned"]
scaler, log_step = lr_pipeline.named_steps["standardscaler"], lr_pipeline.named_steps["functiontransformer"]
train_means = log_step.transform(X_tr).mean(axis=0)                             # what the scaler was shown when it was fitted
pooled_means = log_step.transform(pd.concat([X_tr, X_te])).mean(axis=0)         # what it would hold had it also seen the test rows
scaler_ok = np.allclose(scaler.mean_, train_means) and not np.allclose(scaler.mean_, pooled_means)
print("scaler means equal the training-set means, and differ from train+test means:", scaler_ok)
assert scaler_ok
print("search rows were all training rows: asserted in section 3.1")
print("thresholds: chosen with best_f1_threshold(y_val, ...) in section 4.2; the test set only scored them")
print("test set: read in section 4.2 (scores), 4.3 (bootstrap), and the checks 5.3 and 5.4 (diagnosis only; nothing was changed because of them)")

scaler means equal the training-set means, and differ from train+test means: True
search rows were all training rows: asserted in section 3.1
thresholds: chosen with best_f1_threshold(y_val, ...) in section 4.2; the test set only scored them
test set: read in section 4.2 (scores), 4.3 (bootstrap), and the checks 5.3 and 5.4 (diagnosis only; nothing was changed because of them)


### 5.6 Is every feature available when the prediction is needed?
`newbalanceOrig` and `newbalanceDest` (and the balance errors built from them) only exist **after** the transaction has
settled. A system that has to decide before settlement cannot use them. Here the tuned model is refitted with only what
is known beforehand: the type, the amount and the two old balances.

In [15]:
PRE = ["isTransfer", "amount", "oldbalanceOrg", "oldbalanceDest"]
pre_model = XGBClassifier(**XGB_BASE, **best_xgb).fit(X_tr[PRE], y_train)
pre_score = pre_model.predict_proba(X_te[PRE])[:, 1]
pre_ap = average_precision_score(y_test, pre_score)
full_ap = average_precision_score(y_test, test_scores["XGBoost, tuned"])
display(pd.DataFrame({"features": ["all engineered (needs post-transaction balances)", "known before settlement only"],
                      "test PR-AUC": [full_ap, pre_ap]}).style.format({"test PR-AUC": "{:.4f}"}).hide(axis="index"))
print(f"PR-AUC without the post-transaction information: {pre_ap:.4f} (chance level {y_test.mean():.4f})")

features,test PR-AUC
all engineered (needs post-transaction balances),0.9975
known before settlement only,0.7595


PR-AUC without the post-transaction information: 0.7595 (chance level 0.0030)


### 5.7 Summary of the checks

In [16]:
dup_frauds = int((shared & (y_test == 1)).sum())
checks = pd.DataFrame([
    {"check": "5.1 Features independent of the label", "question": "Does any feature use the label?",
     "result": "features identical after shuffling the label column", "verdict": "ruled out"},
    {"check": "5.2 Single-feature screen", "question": "Does one column almost give the label away?",
     "result": f"strongest single feature: {screen.index[0]} at PR-AUC {screen.iloc[0, 0]:.3f} (computed from the row's own balances)", "verdict": "explained, see note"},
    {"check": "5.3 Duplicate rows", "question": "Is the test score inflated by memorised twins?",
     "result": f"{shared.sum():,} test rows ({shared.mean():.2%}) and {dup_frauds} test frauds have a training twin; tuned PR-AUC without them {twin_tbl.loc[1, 'PR-AUC, twin-free test rows']:.4f} vs {twin_tbl.loc[1, 'PR-AUC, all test rows']:.4f}",
     "verdict": "ruled out" if abs(twin_tbl.loc[1, "change"]) < 0.005 else "affects the score"},
    {"check": "5.4 Shuffled training labels", "question": "Does test information reach the score without the training labels?",
     "result": f"shuffled-label PR-AUC {np.mean(perm):.4f} on average, at most {np.max(perm + perm_shallow):.4f} over {len(perm + perm_shallow)} runs; real score {real_ap:.4f}; chance {y_test.mean():.4f}",
     "verdict": "ruled out" if np.max(perm + perm_shallow) < 0.1 * real_ap else "investigate"},
    {"check": "5.5 Search, threshold and scaling lanes", "question": "Did tuning or preprocessing see validation or test rows?",
     "result": "search rows asserted to be training rows; scaler fitted on training rows only; thresholds from validation", "verdict": "ruled out"},
    {"check": "5.6 Feature availability", "question": "Are all features known when the prediction is needed?",
     "result": f"without post-transaction balances the tuned model scores PR-AUC {pre_ap:.4f} instead of {full_ap:.4f}",
     "verdict": "limits how the model can be used, see note"},
])
checks.to_csv(J2_DIR / "leakage_checks.csv", index=False)
display(checks.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

check,question,result,verdict
5.1 Features independent of the label,Does any feature use the label?,features identical after shuffling the label column,ruled out
5.2 Single-feature screen,Does one column almost give the label away?,strongest single feature: oldbalanceOrg at PR-AUC 0.290 (computed from the row's own balances),"explained, see note"
5.3 Duplicate rows,Is the test score inflated by memorised twins?,11 test rows (0.00%) and 11 test frauds have a training twin; tuned PR-AUC without them 0.9975 vs 0.9975,ruled out
5.4 Shuffled training labels,Does test information reach the score without the training labels?,"shuffled-label PR-AUC 0.0077 on average, at most 0.0538 over 10 runs; real score 0.9975; chance 0.0030",ruled out
"5.5 Search, threshold and scaling lanes",Did tuning or preprocessing see validation or test rows?,search rows asserted to be training rows; scaler fitted on training rows only; thresholds from validation,ruled out
5.6 Feature availability,Are all features known when the prediction is needed?,without post-transaction balances the tuned model scores PR-AUC 0.7595 instead of 0.9975,"limits how the model can be used, see note"


### 5.8 What the checks show

- **Ruled out:** features that use the label (5.1), memorised duplicates inflating the score (5.3), a path from the test
  labels to the score that bypasses the training labels (5.4), and tuning, thresholds or scaling seeing rows they should
  not have (5.5).
- **No feature gives the label away.** The strongest single column reaches a PR-AUC of 0.29 on its own against a naive
  0.003 (5.2). The model's near-perfect score comes from combinations of columns, not from one.
- **Not at chance exactly.** Shuffled-label scores run from 0.002 to 0.054 across ten runs, above the 0.003 chance
  level, and the depth-2 control was not lower. No mechanism is claimed. The conclusion rests on size and direction: the
  highest is 5.4% of the real score, and true frauds do not score higher than legitimate rows under a shuffled fit.
- **A limit on use, not a leak:** the model depends on balances that exist only after a transaction settles (5.6).
  Without them it scores 0.76 instead of 0.9975. It is a post-settlement detector.
- **Not covered here:** a time-based check. J1 ran a chronological hold-out for the baseline (notebook 01, §6.6); it was
  not repeated for the tuned model.

The full write-up is in [reports/j2/leakage_note.md](../reports/j2/leakage_note.md).